# World Bank Crop Price Data Collection

This notebook fetches crop price data from the World Bank's microdata API for Guinea and Liberia, processes it, and exports it to Stata format.


## Setup and Configuration

Initialize required libraries and set up the output file path for the Stata data file.


In [1]:
# Import libraries
import requests
import pandas as pd
import json
from pathlib import Path

# Base path. Change this to your dropbox folder
mypc_path = "/Users/justinblack/Library/CloudStorage/Dropbox"

# Data and output folder. These should stay the same
data_folder = f'{mypc_path}/Lavoro_Replication/0_data_construction'  
output_folder = f'{data_folder}/0_data_input/data_processed/crop_prices'

# Create directory from output folder and name the file
output_file_name = "wbk_crop_prices.csv"
output_path = f"{output_folder}/{output_file_name}"

# For debugging, save locally
# output_path = ""

## API Configuration

Configure the World Bank API parameters:
1. Set up the base URL
2. Define required data fields
3. Specify target countries
4. Set pagination parameters


In [2]:
# Set up the API endpoint from the World Bank 
base_url = "https://microdata.worldbank.org/index.php/api/tables/data/fcv/wld_2021_rtfp_v02_m"

# The World Bank API has many fields and countries; we only want these.
# @ADAGT, out of all the requested fields to add, only a "plantains" field exists in the data. And plaintains has no data for Liberia nor Guinea.
meta_fields = ",".join([
    "country", "lat", "adm1_name", "lon", "adm2_name", "geo_id", "mkt_name",
    "DATES", "year", "month", "currency", "data_coverage", "last_survey_point",
    "components", "start_dense_data", "data_coverage_recent",
    "index_confidence_score", "spatially_interpolated"
])

crop_fields = ["cassava", "maize", "groundnuts", "millet", "rice", 
    "sorghum", "oil", "plantains"]

fields = meta_fields + "," + ",".join(crop_fields)

countries = ["Liberia", "Guinea"]
limit = 100

# Initialize an empty list to store all records
all_records = []

## Data Collection

Fetch data from the World Bank API:
1. Iterate through each country
2. Make paginated API requests
3. Handle response validation and error checking
4. Parse JSON responses
5. Store data in DataFrames


In [3]:
for country in countries:
    print(f"Fetching data for {country}...")
    offset = 0
    while True:
        # Initialize parameters for the API request based on above 
        params = {
            "country": country,
            "fields": fields,
            "limit": limit,
            "offset": offset
        }
        
        # Make the API request
        response = requests.get(base_url, params=params)
        
        # Check if the request was successful
        if response.status_code != 200:
            print(f"Error fetching data for {country} at offset {offset}: {response.status_code}")
            break
        
        # Check the content-type
        content_type = response.headers.get("Content-Type", "")
        if "application/json" not in content_type:
            print(f"Unexpected content-type at offset {offset}: {content_type}")
            break

        # Parse the JSON data safely
        try:
            json_data = json.loads(response.content)
        except json.JSONDecodeError:
            print(f"JSON decode error at offset {offset}")
            break

        # Get the data from the JSON response
        data_raw = json_data.get("data", [])
        if not data_raw:
            print(f"No more data for {country} at offset {offset}")
            break

        # Convert the data to a DataFrame
        df = pd.DataFrame(data_raw)
        all_records.append(df)

        # If the data is less than the limit, break the loop
        if len(data_raw) < limit:
            break  # no more pages

        # Increment the offset
        offset += limit

Fetching data for Liberia...
Fetching data for Guinea...
No more data for Guinea at offset 4200


## Data Processing and Export

Process the collected data and prepare for csv export:
1. Combine all records into a single DataFrame
2. Drop complex columns
3. Convert numeric fields to appropriate types
4. Handle string columns for Stata compatibility
5. Export to .csv format


In [4]:
# Combine all records into a single dataframe
if all_records:
    combined_df = pd.concat(all_records, ignore_index=True)

    # Drop complex columns if present
    for col in combined_df.columns:
        if combined_df[col].apply(lambda x: isinstance(x, (dict, list))).any():
            print(f"Dropping complex column: {col}")
            combined_df = combined_df.drop(columns=[col])

    # Convert relevant fields to float
    for col in ["lat", "lon"] + crop_fields:
        if col in combined_df.columns:
            combined_df[col] = pd.to_numeric(combined_df[col], errors='coerce')

    # Export to Stata by looping through object columns
    for col in combined_df.select_dtypes(include='object').columns:
        unique_types = set(type(x) for x in combined_df[col].dropna())
        if all(t in [str, type(None)] for t in unique_types):
            # Convert to string if valid for Stata
            combined_df[col] = combined_df[col].astype(str)
        elif combined_df[col].isnull().all():
            # Drop columns with only nulls
            combined_df.drop(columns=[col], inplace=True)
        else:
            print(f"⚠️ Column '{col}' has unsupported types: {unique_types}")
            # Either drop or convert based on your needs
            combined_df.drop(columns=[col], inplace=True)

    combined_df.to_csv(output_path, index=False)

    print(f"\nData exported at: {output_path}")
else:
    print("No data retrieved for any country.")


Data exported at: /Users/justinblack/Library/CloudStorage/Dropbox/Lavoro_Replication/0_data_construction/0_data_input/data_processed/crop_prices/wbk_crop_prices.csv
